In [1]:
import pandas as pd

In [2]:
pred = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/invoice_detection_results_12_05_2026_v1_qwen32B.csv")
# pred = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/invoice_detection_results_13_05_2026_v2_qwen32B_startendtokens.csv")

In [3]:
import re
import json
import pandas as pd
import numpy as np

def extract_first_json(text):
    text = (
        text
        .replace(" ","")
        .replace("\n","")
        .replace("True", "true")
        .replace("False", "false")
        .replace("None", "null")
        .replace("'",'"')
    )
    match = re.search(r'\{.*?\}', text)
    if match:
        try:
            json_data = json.loads(match.group())
            return json_data
        except json.JSONDecodeError as e:
            print(match.group())
            print("failed", e)
            return {}
    else:
        print("format failed")
        return {}

def extract_last_json(text):
    text = (
        text
        .replace(" ","")
        .replace("\n","")
        .replace("True", "true")
        .replace("False", "false")
        .replace("None", "null")
        .replace("'",'"')
    )
    # Find all JSON objects in the text
    matches = list(re.finditer(r'\{.*?\}', text))
    if matches:
        # Get the last match
        last_match = matches[-1]
        try:
            json_data = json.loads(last_match.group())
            return json_data
        except json.JSONDecodeError as e:
            print(last_match.group())
            print("failed", e)
            return {}
    else:
        print("format failed")
        return {}
    
def parse_date_safe(date):
    try:
        return pd.to_datetime(date)
    except:
        return np.nan

In [4]:
pred["pred_parsed"] = pred["pred_llm"].apply(extract_last_json)

In [5]:
pred["pred_is_invoice_inside"] = pred["pred_parsed"].apply(lambda x: x.get("is_invoice_inside", False))
pred["pred_start_page"] = pred["pred_parsed"].apply(lambda x: x.get("start_page", False))
pred["pred_end_page"] = pred["pred_parsed"].apply(lambda x: x.get("end_page", False))

In [6]:
pred

,ticket_uuid,attachment_id,invoice_page_start,invoice_page_end,s3_key,s3_bucket,textract_job_id,number_of_pages,source,is_ve_with_invoice,is_invoice_inside,local_file_path,textract_s3_link,clean_text,lentgh,pred_llm,pred_parsed,pred_is_invoice_inside,pred_start_page,pred_end_page
0,21ff29f9-0939-5ba3-b632-b77ea3077483,f0ac7d1c-7e27-5c24-9f8f-7a4e40ed833a,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,ef42e149fb904aafacc40b21da3f74aef2905e52f8271b...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nhelga gerstl\nhauptgerichtsvollziehe...,20043,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
1,ad7254c5-f694-5f0b-8562-6148f0881523,7def55aa-6fcc-5904-b12f-dc882a2ca729,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81764bc01661bb977efafe2719191ce13aee52f98172ae...,9,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieherin\n56290 mack...,19555,"Okay, let's tackle this step by step. The task...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
2,6114ea8c-0415-539d-bbb2-43f2d8d5efcd,0c44f470-a74a-51b1-9e4a-fd47bbf492d8,6.0,6.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,75d7deaabda31af5ee5928ffcb38af9d1c3751f7d28132...,6,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d.:\nogv ko...,13233,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 6, '...",True,6,6
3,8275e126-f6f4-563b-9336-3e4b41dd1522,09d7bb75-16a0-5049-9feb-c6b00372861c,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81afd26f8317b8190a662546f16b303c19a5111ad5f98c...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\nrutesheimer st...,19893,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
4,fa82e542-c3a6-5eea-b252-a9e4e3e8bed3,347853fc-8b9f-522a-a40d-21bf915b1af9,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,c5e37ad57c30062544f7d687aa8b12c44a542b18802eec...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\npoststraße 1\n...,19290,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,c9a67751-117a-511f-a3ba-707465ffccde,60060512,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_349816/600...,pair-data-engineering-new,0770238c1e81a6396322ec541fe565db058334efb7a1fd...,3,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\npatrick dangl\njohn-f.-kennedy-straß...,3287,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1
298,b11e0d18-3d0e-563d-bb7d-3d5f0d394b2a,60062385,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350075/600...,pair-data-engineering-new,304777b40ee39bcbf467dce2b85c7ff328f877b67169ac...,1,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieher\nbeethovenstraße ...,1914,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1
299,49fabc6f-9f0b-5ac0-85fe-57151c9768c2,60062423,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350085/600...,pair-data-engineering-new,aecb51efe65c0eeda6702957181dfd96722de8d9fa065f...,2,prod,NaN,False,/Users/melih.go

In [7]:
gt_invoice_start = pred["invoice_page_start"].apply(lambda x: int(x) if pd.notnull(x) else np.nan)
gt_invoice_end = pred["invoice_page_end"].apply(lambda x: int(x) if pd.notnull(x) else np.nan)

# precision, recall for invoice detection
tp = ((pred["pred_is_invoice_inside"] == True) & (pred["is_invoice_inside"] == True)).sum()
fp = ((pred["pred_is_invoice_inside"] == True) & (pred["is_invoice_inside"] == False)).sum()
fn = ((pred["pred_is_invoice_inside"] == False) & (pred["is_invoice_inside"] == True)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Precision: 0.98
Recall: 0.99


In [8]:
tp_invoice_start = ((pred["pred_start_page"] == gt_invoice_start) & (pred["is_invoice_inside"] == True)).sum()
fp_invoice_start = ((pred["pred_start_page"] != gt_invoice_start) & (pred["is_invoice_inside"] == True)).sum()
fn_invoice_start = ((pred["pred_start_page"] != gt_invoice_start) & (pred["is_invoice_inside"] == True)).sum()

precision_invoice_start = tp_invoice_start / (tp_invoice_start + fp_invoice_start) if (tp_invoice_start + fp_invoice_start) > 0 else 0
recall_invoice_start = tp_invoice_start / (tp_invoice_start + fn_invoice_start) if (tp_invoice_start + fn_invoice_start) > 0 else 0
print(f"Start Page Precision: {precision_invoice_start:.2f}")
print(f"Start Page Recall: {recall_invoice_start:.2f}")

Start Page Precision: 0.95
Start Page Recall: 0.95


In [9]:
tp_invoice_end = ((pred["pred_end_page"] == gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()
fp_invoice_end = ((pred["pred_end_page"] != gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()
fn_invoice_end = ((pred["pred_end_page"] != gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()

precision_invoice_end = tp_invoice_end / (tp_invoice_end + fp_invoice_end) if (tp_invoice_end + fp_invoice_end) > 0 else 0
recall_invoice_end = tp_invoice_end / (tp_invoice_end + fn_invoice_end) if (tp_invoice_end + fn_invoice_end) > 0 else 0
print(f"End Page Precision: {precision_invoice_end:.2f}")
print(f"End Page Recall: {recall_invoice_end:.2f}")

End Page Precision: 0.70
End Page Recall: 0.70


In [10]:
print("Shape:", pred.shape)
print("\nColumns:")
print(pred.columns.tolist())
print("\nDtypes:")
print(pred.dtypes)
print("\nHead:")
pred.head()

Shape: (302, 20)

Columns:
['ticket_uuid', 'attachment_id', 'invoice_page_start', 'invoice_page_end', 's3_key', 's3_bucket', 'textract_job_id', 'number_of_pages', 'source', 'is_ve_with_invoice', 'is_invoice_inside', 'local_file_path', 'textract_s3_link', 'clean_text', 'lentgh', 'pred_llm', 'pred_parsed', 'pred_is_invoice_inside', 'pred_start_page', 'pred_end_page']

Dtypes:
ticket_uuid                object
attachment_id              object
invoice_page_start        float64
invoice_page_end          float64
s3_key                     object
s3_bucket                  object
textract_job_id            object
number_of_pages             int64
source                     object
is_ve_with_invoice         object
is_invoice_inside            bool
local_file_path            object
textract_s3_link           object
clean_text                 object
lentgh                      int64
pred_llm                   object
pred_parsed                object
pred_is_invoice_inside       bool
pred_start_

,ticket_uuid,attachment_id,invoice_page_start,invoice_page_end,s3_key,s3_bucket,textract_job_id,number_of_pages,source,is_ve_with_invoice,is_invoice_inside,local_file_path,textract_s3_link,clean_text,lentgh,pred_llm,pred_parsed,pred_is_invoice_inside,pred_start_page,pred_end_page
0,21ff29f9-0939-5ba3-b632-b77ea3077483,f0ac7d1c-7e27-5c24-9f8f-7a4e40ed833a,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,ef42e149fb904aafacc40b21da3f74aef2905e52f8271b...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nhelga gerstl\nhauptgerichtsvollziehe...,20043,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
1,ad7254c5-f694-5f0b-8562-6148f0881523,7def55aa-6fcc-5904-b12f-dc882a2ca729,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81764bc01661bb977efafe2719191ce13aee52f98172ae...,9,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieherin\n56290 mack...,19555,"Okay, let's tackle this step by step. The task...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
2,6114ea8c-0415-539d-bbb2-43f2d8d5efcd,0c44f470-a74a-51b1-9e4a-fd47bbf492d8,6.0,6.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,75d7deaabda31af5ee5928ffcb38af9d1c3751f7d28132...,6,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d.:\nogv ko...,13233,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 6, '...",True,6,6
3,8275e126-f6f4-563b-9336-3e4b41dd1522,09d7bb75-16a0-5049-9feb-c6b00372861c,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81afd26f8317b8190a662546f16b303c19a5111ad5f98c...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\nrutesheimer st...,19893,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
4,fa82e542-c3a6-5eea-b252-a9e4e3e8bed3,347853fc-8b9f-522a-a40d-21bf915b1af9,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,c5e37ad57c30062544f7d687aa8b12c44a542b18802eec...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\npoststraße 1\n...,19290,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1


# Lets apply some tweaks

# 1) Use Page Numbers in the document

In [11]:
pred

,ticket_uuid,attachment_id,invoice_page_start,invoice_page_end,s3_key,s3_bucket,textract_job_id,number_of_pages,source,is_ve_with_invoice,is_invoice_inside,local_file_path,textract_s3_link,clean_text,lentgh,pred_llm,pred_parsed,pred_is_invoice_inside,pred_start_page,pred_end_page
0,21ff29f9-0939-5ba3-b632-b77ea3077483,f0ac7d1c-7e27-5c24-9f8f-7a4e40ed833a,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,ef42e149fb904aafacc40b21da3f74aef2905e52f8271b...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nhelga gerstl\nhauptgerichtsvollziehe...,20043,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
1,ad7254c5-f694-5f0b-8562-6148f0881523,7def55aa-6fcc-5904-b12f-dc882a2ca729,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81764bc01661bb977efafe2719191ce13aee52f98172ae...,9,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieherin\n56290 mack...,19555,"Okay, let's tackle this step by step. The task...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
2,6114ea8c-0415-539d-bbb2-43f2d8d5efcd,0c44f470-a74a-51b1-9e4a-fd47bbf492d8,6.0,6.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,75d7deaabda31af5ee5928ffcb38af9d1c3751f7d28132...,6,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d.:\nogv ko...,13233,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 6, '...",True,6,6
3,8275e126-f6f4-563b-9336-3e4b41dd1522,09d7bb75-16a0-5049-9feb-c6b00372861c,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81afd26f8317b8190a662546f16b303c19a5111ad5f98c...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\nrutesheimer st...,19893,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
4,fa82e542-c3a6-5eea-b252-a9e4e3e8bed3,347853fc-8b9f-522a-a40d-21bf915b1af9,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,c5e37ad57c30062544f7d687aa8b12c44a542b18802eec...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\npoststraße 1\n...,19290,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,c9a67751-117a-511f-a3ba-707465ffccde,60060512,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_349816/600...,pair-data-engineering-new,0770238c1e81a6396322ec541fe565db058334efb7a1fd...,3,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\npatrick dangl\njohn-f.-kennedy-straß...,3287,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1
298,b11e0d18-3d0e-563d-bb7d-3d5f0d394b2a,60062385,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350075/600...,pair-data-engineering-new,304777b40ee39bcbf467dce2b85c7ff328f877b67169ac...,1,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieher\nbeethovenstraße ...,1914,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1
299,49fabc6f-9f0b-5ac0-85fe-57151c9768c2,60062423,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350085/600...,pair-data-engineering-new,aecb51efe65c0eeda6702957181dfd96722de8d9fa065f...,2,prod,NaN,False,/Users/melih.go

In [12]:
import re


def check_seite_marker_inside(text):
    # german page markers: "Seite 1", "Seite: 1", "Seite 1/3", "Seite 1 von 3", "Blatt 1"
    patterns = [
        r'(?i)\bseite\s*:?\s*\d+',        # Seite 1, Seite: 1
        r'(?i)\bseite\s*\d+\s*/\s*\d+',   # Seite 1/3
        r'(?i)\bseite\s*\d+\s*von\s*\d+', # Seite 1 von 3
        r'(?i)\bblatt\s*:?\s*\d+',        # Blatt 1
    ]
    
    for p in patterns:
        match = re.search(p, text)
        if match:
            # returned matched pattern aswell
            return True, match.group()
    return False, None


def extract_page_text(full_text:str, page_idx:int, number_of_pages:int)-> str:
    page_text = ""
    if page_idx == number_of_pages:
        # last page, no end marker
        start_marker = f"<page_{page_idx}>"
        # get all the text after the start marker
        if start_marker in full_text:
            page_text = full_text.split(start_marker)[-1]
    else:
        start_marker = f"<page_{page_idx}>"
        end_marker = f"<page_{page_idx+1}>"
        # get all the text between the start and end markers
        if start_marker in full_text and end_marker in full_text:
            page_text = full_text.split(start_marker)[-1].split(end_marker)[0]
            
    return page_text

def arg_find_consecutive_sequence_for_invoice(found_page_markers):
    # first one is predicted invoice page, so start from there and find the longest consecutive sequence
    try:
        invoce_page_seite = re.search(r'\d+', found_page_markers[0])
    except IndexError:
        # no page markers found
        return []
    # remove digit to see which pattern it is
    invoce_page_pattern = re.sub(r'\d+', '', found_page_markers[0]).strip().lower()
    for i in range(1, len(found_page_markers)):
        current_marker = found_page_markers[i]
        current_marker_seite = re.search(r'\d+', current_marker)
        current_marker_pattern = re.sub(r'\d+', '', current_marker).strip().lower()
        if current_marker_pattern != invoce_page_pattern:
            # pattern is different, break the sequence
            return [j for j in range(i)]
        if current_marker_seite and invoce_page_seite:
            if int(current_marker_seite.group()) != int(invoce_page_seite.group()) + i:
                # page number is not consecutive, break the sequence
                return [j for j in range(i)]
    return [j for j in range(len(found_page_markers))]

def include_if_page_number_exists(row):
    if row['pred_is_invoice_inside'] != True:
        # dont change anything
        return -1, -1
    # search for page markers
    
    all_page_markers = re.findall(r'<page_(\d+)>', row['clean_text'])
    all_page_numbers = [int(num) for num in all_page_markers]
    n_of_pages = max(all_page_numbers) if all_page_numbers else 0
    
    start_page_idx = row['pred_start_page']
    end_page_idx = row['pred_end_page']
    
    invoce_page_text = extract_page_text(row['clean_text'], start_page_idx, n_of_pages) # invoice text.
    
    cur_text = invoce_page_text # at first iteration, cur_text is text of the predicted invoice start page
    cur_page_idx = start_page_idx # at first iteration, cur_page_idx is the predicted invoice start page index
    found_page_markers = []
    found_page_markers_page_indices = []
    # Iteration forward
    while True:
        # check if there is a page marker in the current text (such as seite 1)
        is_with_page_marker, matched_string = check_seite_marker_inside(cur_text)
        
        # if the marker found, go to next page. if not, break the loop
        if is_with_page_marker and matched_string:
            found_page_markers.append(matched_string)
            found_page_markers_page_indices.append(cur_page_idx)
            cur_page_idx += 1
            cur_text = extract_page_text(row['clean_text'], cur_page_idx, n_of_pages)
            
        else:
            break
        
    indexes = arg_find_consecutive_sequence_for_invoice(found_page_markers)
    if indexes:
        # update the end page index if the algorithm found it
        end_page_idx = found_page_markers_page_indices[indexes[-1]]

    # iteration backward
    # Walk back from the predicted start page while previous pages share the same
    # marker pattern (e.g. "Seite") AND their numbers strictly decrement by 1
    # AND stay > 0. Stop on any mismatch.
    is_with_page_marker, matched_string_predicted_start_page = check_seite_marker_inside(invoce_page_text)
    matched_number = re.search(r'\d+', matched_string_predicted_start_page) if matched_string_predicted_start_page else None
    if is_with_page_marker and matched_string_predicted_start_page and matched_number and int(matched_number.group()) > 1:
        # E.g seite 2 -> check for seite 1 in previos pages
        start_pattern = re.sub(r'\d+', '', matched_string_predicted_start_page).strip().lower()
        expected_number = int(matched_number.group()) - 1
        prev_page_idx = start_page_idx - 1
        while prev_page_idx >= 1 and expected_number > 0:
            prev_page_text = extract_page_text(row['clean_text'], prev_page_idx, n_of_pages)
            is_with_prev_marker, matched_previous_page = check_seite_marker_inside(prev_page_text)
            if not (is_with_prev_marker and matched_previous_page):
                # no marker on previous page -> stop
                break

            prev_pattern = re.sub(r'\d+', '', matched_previous_page).strip().lower()
            if prev_pattern != start_pattern:
                # different pattern family (e.g. Seite -> Blatt) -> stop
                break

            prev_number_match = re.search(r'\d+', matched_previous_page)
            if not prev_number_match:
                break
            prev_number = int(prev_number_match.group())
            if prev_number != expected_number:
                # numbering not consecutively decreasing -> stop
                break

            # valid previous page belonging to the same invoice -> extend backward
            start_page_idx = prev_page_idx
            expected_number -= 1
            prev_page_idx -= 1

    return start_page_idx, end_page_idx



In [13]:
_corrected = pred.apply(include_if_page_number_exists, axis=1)

pred['new_start'] = _corrected.apply(lambda t: t[0])
pred['new_would'] = _corrected.apply(lambda t: t[1])

In [14]:
# calculate precision recall with new end page predictions
tp_invoice_end_new = ((pred["new_would"] == gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()
fp_invoice_end_new = ((pred["new_would"] != gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()
fn_invoice_end_new = ((pred["new_would"] != gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()

precision_invoice_end_new = tp_invoice_end_new / (tp_invoice_end_new + fp_invoice_end_new) if (tp_invoice_end_new + fp_invoice_end_new) > 0 else 0
recall_invoice_end_new = tp_invoice_end_new / (tp_invoice_end_new + fn_invoice_end_new) if (tp_invoice_end_new + fn_invoice_end_new) > 0 else 0
print(f"New End Page Precision: {precision_invoice_end_new:.2f}")
print(f"New End Page Recall: {recall_invoice_end_new:.2f}")

New End Page Precision: 0.91
New End Page Recall: 0.91


In [15]:
# start page accuracy after backward iteration
mask = pred["is_invoice_inside"] == True
acc_start_new = (pred.loc[mask, "new_start"] == gt_invoice_start[mask]).mean()
acc_start_old = (pred.loc[mask, "pred_start_page"] == gt_invoice_start[mask]).mean()
print(f"Original Start Page Accuracy: {acc_start_old:.2f}")
print(f"New Start Page Accuracy:      {acc_start_new:.2f}")


Original Start Page Accuracy: 0.95
New Start Page Accuracy:      0.96


# 2) If next page lenght is small for some threshold, include it as well

In [16]:
pred

,ticket_uuid,attachment_id,invoice_page_start,invoice_page_end,s3_key,s3_bucket,textract_job_id,number_of_pages,source,is_ve_with_invoice,...,textract_s3_link,clean_text,lentgh,pred_llm,pred_parsed,pred_is_invoice_inside,pred_start_page,pred_end_page,new_start,new_would
0,21ff29f9-0939-5ba3-b632-b77ea3077483,f0ac7d1c-7e27-5c24-9f8f-7a4e40ed833a,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,ef42e149fb904aafacc40b21da3f74aef2905e52f8271b...,8,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nhelga gerstl\nhauptgerichtsvollziehe...,20043,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
1,ad7254c5-f694-5f0b-8562-6148f0881523,7def55aa-6fcc-5904-b12f-dc882a2ca729,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81764bc01661bb977efafe2719191ce13aee52f98172ae...,9,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieherin\n56290 mack...,19555,"Okay, let's tackle this step by step. The task...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
2,6114ea8c-0415-539d-bbb2-43f2d8d5efcd,0c44f470-a74a-51b1-9e4a-fd47bbf492d8,6.0,6.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,75d7deaabda31af5ee5928ffcb38af9d1c3751f7d28132...,6,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d.:\nogv ko...,13233,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 6, '...",True,6,6,6,6
3,8275e126-f6f4-563b-9336-3e4b41dd1522,09d7bb75-16a0-5049-9feb-c6b00372861c,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81afd26f8317b8190a662546f16b303c19a5111ad5f98c...,8,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\nrutesheimer st...,19893,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
4,fa82e542-c3a6-5eea-b252-a9e4e3e8bed3,347853fc-8b9f-522a-a40d-21bf915b1af9,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,c5e37ad57c30062544f7d687aa8b12c44a542b18802eec...,8,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\npoststraße 1\n...,19290,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,c9a67751-117a-511f-a3ba-707465ffccde,60060512,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_349816/600...,pair-data-engineering-new,0770238c1e81a6396322ec541fe565db058334efb7a1fd...,3,prod,NaN,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\npatrick dangl\njohn-f.-kennedy-straß...,3287,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
298,b11e0d18-3d0e-563d-bb7d-3d5f0d394b2a,60062385,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350075/600...,pair-data-engineering-new,304777b40ee39bcbf467dce2b85c7ff328f877b67169ac...,1,prod,NaN,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieher\nbeethovenstraße ...,1914,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
299,49fabc6f-9f0b-5ac0-85fe-57151c9768c2,60062423,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350085/600...,pair-data-engineering-new,aecb51efe65c0eeda6702957181dfd96722de8d9fa065f...,2,prod,NaN,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nm. jödicke\nhellersdorfer weg 35\nob...,2298,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
300,fa8e0a2c-8d31-5b7a-92e4-99c2a4542f8a,60062417,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350083/600...,pair-data-engin

In [17]:
multiple_invoice_pages = pred[(pred['is_invoice_inside'] == True) & ((pred['invoice_page_end'] - pred['invoice_page_start']) > 0)]
multiple_invoice_pages

,ticket_uuid,attachment_id,invoice_page_start,invoice_page_end,s3_key,s3_bucket,textract_job_id,number_of_pages,source,is_ve_with_invoice,...,textract_s3_link,clean_text,lentgh,pred_llm,pred_parsed,pred_is_invoice_inside,pred_start_page,pred_end_page,new_start,new_would
5,37e5aa21-b9ac-5544-b67c-defb0d02d34b,2ad6769d-9f53-5fd3-97d3-e107d3496354,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,45b04d9b416a81516fe297befef8bf95076c410bb5dcdd...,10,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieher beim amtsgericht ...,21039,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,2
13,23f0c02a-6907-54dd-9bb0-1af425125532,5fa8a38e-4ec9-5416-a38e-4c070416cf5c,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,cd0c089494f31d0d2b1c6959adfac5e93aecbd48b2c212...,9,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieher\nam kalkberg ...,19826,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 2, '...",True,2,2,2,2
14,2776c8cd-33cc-5f86-afbb-dcd207cc0fa1,f120e39d-6b59-532f-8c94-0c163fcda0ee,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,3bb905df995a7a12e8084cf8cd313d2032d8bb5d7b096f...,9,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nreiner herrling\nhauptgerichtsvollzi...,19734,"Okay, I need to analyze the provided German do...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,2,1,2
15,63e055f7-c3cb-5cf5-ad01-5837b73e9dc6,fec6bb35-9237-512d-ba5d-485dd1bdc51c,6.0,7.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,1237199e2c96448d7109a0afc50add09809429f4583067...,7,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d. obergeri...,13370,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 7, '...",True,7,7,7,7
16,2feef0cd-88c6-5b8f-92b5-3c0a322f75b6,4f4ea431-0fb7-586f-bb27-9463d7cb820b,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,a528920231f4faa574afc9000decc5effc72b5510ebc6d...,9,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\nschöttlinstr. ...,19270,"Okay, let me tackle this problem step by step....","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,2
18,a955cc6d-9140-592d-9498-6865c5ea4669,12d88ca5-5646-5009-b999-b5bc45857f4d,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,15a7ba7502aa20639b20bcaf5b4d116c9e6fa18147dc33...,6,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nalina greve\npostfach 12 27\ngericht...,12823,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,2
20,228c2a2e-27f2-507f-8f51-0ca06c614595,dc016285-7db1-5351-bbee-d78c24c4aadd,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,8accd0ec02fde25b7d079e6f70904df510476e79dbe81a...,9,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieherin paul\ndiens...,20926,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
21,e350d6ee-227e-5860-a882-14ed99281539,44479a3a-5d8d-5a9b-9c86-c0b13d1c99e0,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,7faf57c87fef13cc6635cc5a5144039eed147705c532b2...,9,raw,True,...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieher\namtsgericht:...,19442,"Okay, let me tackle this problem step by step....","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,2
22,d4ff1227-7b83-551a-8814-69c1316a8c83,a04fc061-a563-5f0f-a2cc-76947db4b816,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pa

In [18]:
second_page_texts = []
for idx, row in multiple_invoice_pages.iterrows():
    clean_text = row['clean_text']
    end_page_idx = int(row['invoice_page_end'])
    all_page_markers = re.findall(r'<page_(\d+)>', clean_text)
    all_page_numbers = [int(num) for num in all_page_markers]
    n_of_pages = max(all_page_numbers) if all_page_numbers else 0
    
    end_page_text = extract_page_text(clean_text, end_page_idx, n_of_pages)
    second_page_texts.append(end_page_text)

In [19]:
lengths = [len(text) for text in second_page_texts]
word_counts = [len(text.split()) for text in second_page_texts]

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=word_counts,
    nbinsx=20,
    marker=dict(
        color="steelblue",
        line=dict(color="white", width=0.5),
    ),
    opacity=0.85,
    hovertemplate="Length: %{x}<br>Count: %{y}<extra></extra>",
))

fig.add_vline(
    x=np.median(word_counts),
    line_dash="dash",
    line_color="tomato",
    annotation_text=f"Median: {int(np.median(word_counts))}",
    annotation_position="top right",
    annotation_font_color="tomato",
)

fig.update_layout(
    title="Distribution of Text Word Counts on Invoice End Pages",
    xaxis_title="Text Word Count",
    yaxis_title="Frequency",
    bargap=0.05,
    template="plotly_white",
    width=800,
    height=450,
)

fig.show()


In [20]:
# be conservative, th = 200


GUARD_PATTERN = r"(?im)^\s*(?:protokoll|verm[oö0]gens\s*verzeichnis|verm[oö0]gens\s*auskunft(?:s\s*protokoll|protokoll)?|dritt\s*ausk[uü]nfte|ergebnis(?:se)?(?:\s+der\s+verm[oö0]gens\s*auskunft)?)\b"
GUARD_RE = re.compile(GUARD_PATTERN)

def include_if_next_page_short(row):
    if row['pred_is_invoice_inside'] != True:
        # dont change anything
        return -1
    
    start_page_idx = row['pred_start_page']
    end_page_idx = row['pred_end_page']
    if start_page_idx != end_page_idx:
        # do not check the next page if the predicted start and end pages are different
        return end_page_idx
    next_page_idx = start_page_idx + 1
    clean_text = row['clean_text']
    # find number of pages in the document
    all_page_markers = re.findall(r'<page_(\d+)>', clean_text)
    all_page_numbers = [int(num) for num in all_page_markers]
    n_of_pages = max(all_page_numbers) if all_page_numbers else 0
    
    # extract page text for the next page
    next_page_text = extract_page_text(clean_text, next_page_idx, n_of_pages)
    word_count = len(next_page_text.split())
    if next_page_text != '' and word_count < 100:
        if GUARD_RE.search(next_page_text):
            # if the next page contains the guard pattern, it is likely that the next page is not part of the invoice, so do not change the end page index
            return end_page_idx
        return next_page_idx
    else:        
        return end_page_idx
    

In [21]:
def apply_both_algorithms(row):
    if row['pred_is_invoice_inside'] != True:
        # dont change anything
        return -1, -1

    start_page_idx, end_page_idx = include_if_page_number_exists(row)
    if end_page_idx == row['pred_end_page']:
        # if the first algorithm did not change the end page, apply the second algorithm
        return start_page_idx, include_if_next_page_short(row)
    else:
        return start_page_idx, end_page_idx


_corrected = pred.apply(apply_both_algorithms, axis=1)

pred['corrected_start_page'] = _corrected.apply(lambda t: t[0])
pred['corrected_end_page'] = _corrected.apply(lambda t: t[1])

In [22]:
# calculate precision recall with new end page predictions

tp_invoice_end_new = ((pred["corrected_end_page"] == gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()
fp_invoice_end_new = ((pred["corrected_end_page"] != gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()
fn_invoice_end_new = ((pred["corrected_end_page"] != gt_invoice_end) & (pred["is_invoice_inside"] == True)).sum()

precision_invoice_end_new = tp_invoice_end_new / (tp_invoice_end_new + fp_invoice_end_new) if (tp_invoice_end_new + fp_invoice_end_new) > 0 else 0
recall_invoice_end_new = tp_invoice_end_new / (tp_invoice_end_new + fn_invoice_end_new) if (tp_invoice_end_new + fn_invoice_end_new) > 0 else 0
print(f"New End Page Precision: {precision_invoice_end_new:.2f}")
print(f"New End Page Recall: {recall_invoice_end_new:.2f}")

New End Page Precision: 0.97
New End Page Recall: 0.97


In [23]:
# start page accuracy after combined algorithms
mask = pred["is_invoice_inside"] == True
acc_start_combined = (pred.loc[mask, "corrected_start_page"] == gt_invoice_start[mask]).mean()
acc_start_old = (pred.loc[mask, "pred_start_page"] == gt_invoice_start[mask]).mean()
print(f"Original Start Page Accuracy: {acc_start_old:.2f}")
print(f"Corrected Start Page Accuracy: {acc_start_combined:.2f}")


Original Start Page Accuracy: 0.95
Corrected Start Page Accuracy: 0.96


In [24]:
# pred['pred_end_page']= pred['corrected_end_page']
# pred['pred_start_page']= pred['corrected_start_page']
# pred.to_csv("60060387.csv", index=False)60060387

# Final Report: Before vs After Heuristics

Comparison of LLM-only predictions vs. predictions after applying the page-marker heuristic and the short-next-page heuristic.


In [25]:
def _accuracy(pred_col, gt_series, mask):
    return (pred_col[mask] == gt_series[mask]).mean()

# --- Invoice detection (binary) ---
det_tp = ((pred["pred_is_invoice_inside"] == True) & (pred["is_invoice_inside"] == True)).sum()
det_fp = ((pred["pred_is_invoice_inside"] == True) & (pred["is_invoice_inside"] == False)).sum()
det_fn = ((pred["pred_is_invoice_inside"] == False) & (pred["is_invoice_inside"] == True)).sum()
det_precision = det_tp / (det_tp + det_fp) if (det_tp + det_fp) else 0
det_recall    = det_tp / (det_tp + det_fn) if (det_tp + det_fn) else 0
det_f1 = (2 * det_precision * det_recall / (det_precision + det_recall)) if (det_precision + det_recall) else 0

# --- Page predictions: only on rows that truly contain an invoice ---
mask = pred["is_invoice_inside"] == True
n_invoices = int(mask.sum())

rows = [
    {
        "Metric": "Invoice detection precision",
        "Before (LLM)": f"{det_precision:.3f}",
        "After (heuristics)": "—",
        "Δ": "—",
    },
    {
        "Metric": "Invoice detection recall",
        "Before (LLM)": f"{det_recall:.3f}",
        "After (heuristics)": "—",
        "Δ": "—",
    },
    {
        "Metric": "Invoice detection F1",
        "Before (LLM)": f"{det_f1:.3f}",
        "After (heuristics)": "—",
        "Δ": "—",
    },
]

def _add_row(label, before_val, after_val):
    delta = after_val - before_val
    rows.append({
        "Metric": label,
        "Before (LLM)": f"{before_val:.3f}",
        "After (heuristics)": f"{after_val:.3f}",
        "Δ": f"{delta:+.3f}",
    })

# Start page
start_before = _accuracy(pred["pred_start_page"], gt_invoice_start, mask)
start_after_alg1 = _accuracy(pred["new_start"], gt_invoice_start, mask)
start_after_combined = _accuracy(pred["corrected_start_page"], gt_invoice_start, mask)
_add_row("Start page accuracy (include_if_page_number_exists)", start_before, start_after_alg1)
_add_row("Start page accuracy (apply_both_algorithms)",         start_before, start_after_combined)

# End page
end_before = _accuracy(pred["pred_end_page"], gt_invoice_end, mask)
end_after_alg1 = _accuracy(pred["new_would"], gt_invoice_end, mask)
end_after_combined = _accuracy(pred["corrected_end_page"], gt_invoice_end, mask)
_add_row("End page accuracy (include_if_page_number_exists)", end_before, end_after_alg1)
_add_row("End page accuracy (apply_both_algorithms)",         end_before, end_after_combined)

# Exact-range match (start AND end correct simultaneously)
range_before = (
    (pred.loc[mask, "pred_start_page"] == gt_invoice_start[mask]) &
    (pred.loc[mask, "pred_end_page"]   == gt_invoice_end[mask])
).mean()
range_after = (
    (pred.loc[mask, "corrected_start_page"] == gt_invoice_start[mask]) &
    (pred.loc[mask, "corrected_end_page"]   == gt_invoice_end[mask])
).mean()
_add_row("Exact range match (start AND end)", range_before, range_after)

report_df = pd.DataFrame(rows)
print(f"Total rows: {len(pred)} | Rows with invoice: {n_invoices}")
report_df


Total rows: 302 | Rows with invoice: 154


,Metric,Before (LLM),After (heuristics),Δ
0,Invoice detection precision,0.981,—,—
1,Invoice detection recall,0.994,—,—
2,Invoice detection F1,0.987,—,—
3,Start page accuracy (include_if_page_number_ex...,0.948,0.961,+0.013
4,Start page accuracy (apply_both_algorithms),0.948,0.961,+0.013
5,End page accuracy (include_if_page_number_exists),0.701,0.909,+0.208
6,End page accuracy (apply_both_algorithms),0.701,0.974,+0.273
7,Exact range match (start AND end),0.656,0.942,+0.286
